In [2]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# ============================================================================
# Project paths
# ============================================================================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR = RESULTS_DIR / 'model'
TABLE_DIR = RESULTS_DIR / 'table'
FIGURE_DIR = RESULTS_DIR / 'figures'

for d in [DATA_DIR, MODEL_DIR, TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project paths ready (data/, results/model/, results/table/, results/figures/)")

# ============================================================================
# 1. Initial Setup and Data Loading
# ============================================================================
file_path = DATA_DIR / '202207_corpor_CB.csv'
df = pd.read_csv(file_path)

feature_cols = [
    'FN1_1', 'FN1_2', 'FN1_3', 'FN1_4', 'FN1_5', 'FN1_6', 'FN1_7', 'FN1_8',
    'FN1_9', 'FN1_10', 'FN1_11', 'FN1_11_1', 'FN1_11_2', 'FN1_11_3', 'FN1_11_4',
    'FN1_13', 'FN1_13_1', 'FN1_14', 'FN1_15', 'FN1_16', 'FN1_17', 'FN1_18',
    'FN1_19', 'FN1_20', 'FN1_21', 'FN1_21_1', 'FN1_22', 'FN1_22_1', 'FN1_22_2',
    'FN1_23', 'FN1_24', 'FN1_24_1', 'FN2_1', 'FN2_1_1', 'FN2_2', 'FN2_2_1',
    'FN2_3', 'FN2_3_1', 'FN2_3_2', 'FN2_3_3', 'FN2_3_4', 'FN2_3_5', 'FN2_4',
    'FN2_5', 'FN2_5_1', 'FN2_7', 'FN2_8', 'FN2_9', 'FN2_10', 'FN2_10_1',
    'FN3_1', 'FN3_2', 'FN3_2_1', 'FN3_2_2', 'FN3_3', 'FN3_4_1', 'FN3_4_2',
    'FN3_6', 'FN3_7', 'FN3_8', 'FN3_10', 'FN3_10_1', 'FN3_11', 'FN3_11_1'
]

id_col = 'ID'
target_col = 'PERF_12M'
selected_cols = [id_col] + feature_cols + [target_col]

print("Data loaded successfully:", df.shape)

# ============================================================================
# 2. SENTINEL VALUE CLEANUP — must run BEFORE any other processing.
#
# Diagnostic finding: the raw dataset contains a sentinel/error value of
# approximately -7.77778e13, appearing (with varying frequency, 0.003%-6.62%
# of rows) across nearly every one of the 64 independent variables. This
# value is orders of magnitude outside any financially plausible range and
# is almost certainly a missing-data placeholder encoded upstream (e.g. an
# original "-77777" or "-99999" style sentinel that was later rescaled).
# Left untreated, this value: (a) propagates into ratio-denominator features
# (e.g. FN1_13, total assets) and distorts every ratio derived from them,
# (b) defeats the accounting-validity filter in Step 3 below (a sentinel
# masquerading as "assets <= 0" or similar), and (c) gets treated by DiCE's
# genetic algorithm as a legitimate candidate value during counterfactual
# optimization, producing physically nonsensical CF outputs (verified: a
# firm's original FN3_3 = 2.30 was replaced by DiCE with -7.77778e13 in a
# generated CF, because that exact value already existed elsewhere in the
# training distribution).
# ============================================================================
print("\n" + "=" * 80)
print("STEP 0 (new): Sentinel Value Detection and Cleanup")
print("=" * 80)

SENTINEL_VALUE = -7.77778e13
SENTINEL_REL_TOL = 1e-3  # relative tolerance for matching, guards against
                          # minor floating-point representation differences

sentinel_report = []
for col in feature_cols:
    is_sentinel = np.isclose(df[col], SENTINEL_VALUE, rtol=SENTINEL_REL_TOL)
    n_sentinel = is_sentinel.sum()
    if n_sentinel > 0:
        sentinel_report.append({'column': col, 'n_sentinel': n_sentinel,
                                 'pct_sentinel': n_sentinel / len(df) * 100})
        # Replace sentinel with NaN so downstream steps treat it as missing,
        # not as a genuine (extreme) observation.
        df.loc[is_sentinel, col] = np.nan

sentinel_report_df = pd.DataFrame(sentinel_report).sort_values('n_sentinel', ascending=False)
print(f"\nColumns affected by sentinel contamination: {len(sentinel_report_df)} / {len(feature_cols)}")
print(sentinel_report_df.to_string(index=False))

sentinel_report_df.to_csv(TABLE_DIR / 'sentinel_contamination_report.csv', index=False, encoding='utf-8-sig')

# ============================================================================
# 3. Impute the newly-created NaNs.
#
# Rationale for median imputation: sentinel contamination affects each
# column at a low rate (median ~0.1-0.6%, max 6.62% for FN3_3), so replacing
# with the column median introduces minimal distortion while avoiding the
# larger information loss of dropping ~10,000 rows (which would disproportio-
# nately remove firms whose OTHER variables are perfectly valid). This
# mirrors the imputation approach already used for division-by-zero NaNs
# in the ratio-conversion step below.
# ============================================================================
print("\n" + "=" * 80)
print("Imputing sentinel-derived missing values (column median)")
print("=" * 80)

for col in feature_cols:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

n_remaining_sentinel = sum(
    np.isclose(df[col], SENTINEL_VALUE, rtol=SENTINEL_REL_TOL).sum() for col in feature_cols
)
print(f"Sentinel values remaining after cleanup: {n_remaining_sentinel} (should be 0)")

# ============================================================================
# 4. No industry filtering — use the full corpus.
# ============================================================================
print("\n" + "=" * 80)
print("Full-Corpus Overview (no industry / audit-class filtering)")
print("=" * 80)

full_df = df[selected_cols].copy()

n_bankrupt = full_df[target_col].sum()
n_total = len(full_df)
print(f"Total firms: {n_total}")
print(f"Bankrupt firms: {n_bankrupt} ({n_bankrupt/n_total*100:.2f}%)")
print(f"Solvent firms: {n_total - n_bankrupt}")
print(f"Class ratio (neg:pos): {(n_total - n_bankrupt)}:{n_bankrupt} = 1:{(n_total - n_bankrupt)/n_bankrupt:.1f}")

n_missing = full_df.isnull().sum().sum()
if n_missing == 0:
    print("\nMissing value check: None (Clean Data)")
else:
    print(f"\n[Warning] {n_missing} missing values detected.")
    print(full_df.isnull().sum()[full_df.isnull().sum() > 0])

sector_composition = df.groupby(['WG_GB', 'SIC_CD_3']).agg({target_col: ['sum', 'count', 'mean']})
sector_composition.columns = ['Bankruptcy_Count', 'Total_Count', 'Bankruptcy_Rate']
sector_composition = sector_composition.sort_values('Bankruptcy_Count', ascending=False)
sector_composition.reset_index().to_csv(
    TABLE_DIR / 'sector_composition_reference.csv', index=False, encoding='utf-8-sig'
)
print(f"\nSector composition saved for reference only: results\\table\\sector_composition_reference.csv")

# ============================================================================
# 5. STEP A — Remove accounting-impossible rows
#    (now runs on sentinel-cleaned data, so this filter is meaningful)
# ============================================================================
print("\n" + "=" * 80)
print("STEP A: Removing accounting-impossible rows")
print("=" * 80)

DENOM_ASSETS = 'FN1_13'
DENOM_REVENUE = 'FN2_1'

n_before = len(full_df)

mask_valid_assets = full_df[DENOM_ASSETS] > 0
mask_equity_le_assets = full_df['FN1_20'] <= full_df[DENOM_ASSETS]
mask_valid_revenue = full_df[DENOM_REVENUE] >= 0

valid_mask = mask_valid_assets & mask_equity_le_assets & mask_valid_revenue

print(f"Removed for assets<=0:            {(~mask_valid_assets).sum()}")
print(f"Removed for equity>assets:        {(~mask_equity_le_assets & mask_valid_assets).sum()}")
print(f"Removed for revenue<0:            {(~mask_valid_revenue & mask_valid_assets & mask_equity_le_assets).sum()}")
print(f"Total removed:                    {(~valid_mask).sum()} / {n_before}")

df_clean = full_df[valid_mask].copy().reset_index(drop=True)
print(f"\nRemaining shape: {df_clean.shape}")
print(f"Remaining bankruptcy count: {df_clean[target_col].sum()} "
      f"(was {full_df[target_col].sum()}, lost {full_df[target_col].sum() - df_clean[target_col].sum()})")

# ============================================================================
# 6. STEP B — Convert absolute-value features to ratios
# ============================================================================
print("\n" + "=" * 80)
print("STEP B: Converting absolute values to ratios")
print("=" * 80)

EPS = 1e-6

GROUP_A_RATIOS_AS_IS = ['FN3_3', 'FN3_6', 'FN3_10']

GROUP_B_TO_ASSET_RATIO = [
    'FN1_1', 'FN1_2', 'FN1_3', 'FN1_4', 'FN1_5', 'FN1_6', 'FN1_7', 'FN1_8',
    'FN1_9', 'FN1_10', 'FN1_11', 'FN1_11_2', 'FN1_11_3', 'FN1_11_4',
    'FN1_14', 'FN1_15', 'FN1_16', 'FN1_17', 'FN1_18', 'FN1_19',
    'FN1_20', 'FN1_21', 'FN1_21_1', 'FN1_22', 'FN1_22_1', 'FN1_22_2',
    'FN1_23', 'FN1_24', 'FN3_10_1', 'FN3_11', 'FN3_11_1',
]

GROUP_C_TO_REVENUE_RATIO = [
    'FN2_2', 'FN2_2_1', 'FN2_3', 'FN2_3_1', 'FN2_3_2', 'FN2_3_3',
    'FN2_3_4', 'FN2_3_5', 'FN2_4', 'FN2_5', 'FN2_5_1', 'FN2_7', 'FN2_8',
    'FN2_9', 'FN2_10', 'FN3_1', 'FN3_2', 'FN3_2_1', 'FN3_2_2',
    'FN3_4_1', 'FN3_4_2', 'FN3_7', 'FN3_8',
]

GROUP_D_GROWTH_PAIRS = [
    ('FN1_13', 'FN1_13_1', 'asset_growth_rate'),
    ('FN2_1', 'FN2_1_1', 'revenue_growth_rate'),
    ('FN2_5', 'FN2_5_1', 'operating_income_growth'),
    ('FN2_10', 'FN2_10_1', 'net_income_growth'),
    ('FN1_24', 'FN1_24_1', 'equity_growth_rate'),
]

ratio_df = pd.DataFrame(index=df_clean.index)
ratio_df[id_col] = df_clean[id_col]

for col in GROUP_A_RATIOS_AS_IS:
    ratio_df[col] = df_clean[col]

assets = df_clean[DENOM_ASSETS].abs() + EPS
for col in GROUP_B_TO_ASSET_RATIO:
    ratio_df[f'{col}_to_assets'] = df_clean[col] / assets

revenue = df_clean[DENOM_REVENUE].replace(0, np.nan).abs() + EPS
for col in GROUP_C_TO_REVENUE_RATIO:
    ratio_df[f'{col}_to_revenue'] = df_clean[col] / revenue

for current_col, prior_col, new_name in GROUP_D_GROWTH_PAIRS:
    prior_val = df_clean[prior_col].replace(0, np.nan).abs() + EPS
    ratio_df[new_name] = (df_clean[current_col] - df_clean[prior_col]) / prior_val

ratio_df[target_col] = df_clean[target_col]

feature_cols_ratio = [c for c in ratio_df.columns if c not in [id_col, target_col]]

n_nan = ratio_df[feature_cols_ratio].isnull().sum().sum()
print(f"NaN introduced by division: {n_nan}")

ratio_df[feature_cols_ratio] = ratio_df[feature_cols_ratio].replace([np.inf, -np.inf], np.nan)
for col in feature_cols_ratio:
    ratio_df[col] = ratio_df[col].fillna(ratio_df[col].median())

print(f"Ratio feature count: {len(feature_cols_ratio)}")
print(f"Shape after ratio conversion: {ratio_df.shape}")

# ============================================================================
# 7. STEP C — Winsorize at 1st/99th percentile (per column)
# ============================================================================
print("\n" + "=" * 80)
print("STEP C: Winsorizing at 1st/99th percentile")
print("=" * 80)

ratio_df_winsorized = ratio_df.copy()
winsor_log = []

for col in feature_cols_ratio:
    p01 = ratio_df[col].quantile(0.01)
    p99 = ratio_df[col].quantile(0.99)
    n_clipped_low = (ratio_df[col] < p01).sum()
    n_clipped_high = (ratio_df[col] > p99).sum()
    ratio_df_winsorized[col] = ratio_df[col].clip(lower=p01, upper=p99)
    winsor_log.append({'column': col, 'p01': p01, 'p99': p99,
                        'n_clipped_low': n_clipped_low, 'n_clipped_high': n_clipped_high})

winsor_summary = pd.DataFrame(winsor_log)
total_clipped = winsor_summary['n_clipped_low'].sum() + winsor_summary['n_clipped_high'].sum()
print(f"Total values clipped across all columns: {total_clipped}")

winsor_summary.to_csv(TABLE_DIR / 'winsorization_log.csv', index=False, encoding='utf-8-sig')

# ============================================================================
# 8. Verification — confirm the FN3_3-style extreme values are gone
# ============================================================================
print("\n" + "=" * 80)
print("VERIFICATION: post-cleanup extreme value check")
print("=" * 80)
for col in GROUP_A_RATIOS_AS_IS:
    print(f"{col}: min={ratio_df_winsorized[col].min():.4f}, max={ratio_df_winsorized[col].max():.4f}")

# ============================================================================
# 9. Save final preprocessed dataset
# ============================================================================
print("\n" + "=" * 80)
print("File Export Process")
print("=" * 80)

output_path = DATA_DIR / 'selected_data_for_modeling_full_ratio_clean.csv'
ratio_df_winsorized.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"1. Dataset saved: {output_path.relative_to(PROJECT_ROOT)}")
print(f"   - Shape: {ratio_df_winsorized.shape}")
print(f"   - Bankruptcy cases included: {ratio_df_winsorized[target_col].sum()}")

pkl_path = DATA_DIR / 'feature_names_full_ratio_clean.pkl'
joblib.dump(feature_cols_ratio, pkl_path)
print(f"2. Feature name list saved: {pkl_path.relative_to(PROJECT_ROOT)}")
print(f"   - Number of features saved: {len(feature_cols_ratio)}")

print("\n[Step 1 Complete] Proceed to the next step (Step2_Modeling).")

Project paths ready (data/, results/model/, results/table/, results/figures/)
Data loaded successfully: (150000, 160)

STEP 0 (new): Sentinel Value Detection and Cleanup

Columns affected by sentinel contamination: 2 / 64
column  n_sentinel  pct_sentinel
 FN3_3        9934      6.622667
 FN3_6          24      0.016000

Imputing sentinel-derived missing values (column median)
Sentinel values remaining after cleanup: 0 (should be 0)

Full-Corpus Overview (no industry / audit-class filtering)
Total firms: 150000
Bankrupt firms: 2154 (1.44%)
Solvent firms: 147846
Class ratio (neg:pos): 147846:2154 = 1:68.6

Missing value check: None (Clean Data)

Sector composition saved for reference only: results\table\sector_composition_reference.csv

STEP A: Removing accounting-impossible rows
Removed for assets<=0:            12
Removed for equity>assets:        2166
Removed for revenue<0:            98
Total removed:                    2276 / 150000

Remaining shape: (147724, 66)
Remaining bankruptc